In [28]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import CITY_COUNTRY, WEATHER_FILES, weather_path

#### Step 1: Load and stack all 11 weather files
**Setup:** imports pandas and our shared settings from `src/config.py`.

`weather_path("Lahore")` gives `Dataset/Pakistan/Weather/Lahore - Weather.csv`

In [29]:
missing = [c for c in WEATHER_FILES if not (ROOT / weather_path(c)).exists()]
print("Missing files:", missing if missing else "none, all 11 found")

for country in ["Pakistan", "India"]:
    folder = ROOT / "Dataset" / country / "Weather"
    print(country, sorted(p.name for p in folder.iterdir()))

Missing files: none, all 11 found
Pakistan ['Faisalabad - Weather.csv', 'Islamabad - Weather.csv', 'Lahore - Weather.csv', 'Multan - Weather.csv', 'Rahim Yar Khan - Weather.csv', 'Rawalpindi weather.csv', 'Sailkot-Weather.csv']
India ['Amritsar - Weather.csv', 'Ferozepur - Weather.csv', 'Jalandhar - Weather.csv', 'Ludhiana - Weather.xlsx']


**Check files exist:** makes sure all 11 files are where the config says, so a wrong name fails here.

In [30]:
RENAME = {
    "temperature_2m (°C)":      "temp_c",
    "relative_humidity_2m (%)": "rh_pct",
    "precipitation (mm)":       "precip_mm",
    "wind_speed_10m (km/h)":    "wind_speed_kmh",
    "wind_direction_10m (°)":   "wind_dir_deg",
}

def load_weather_file(city):
    path = ROOT / weather_path(city)
    reader = pd.read_excel if path.suffix == ".xlsx" else pd.read_csv

    meta = reader(path, nrows=1)        # header block: latitude, longitude, timezone
    df = reader(path, skiprows=3)       # skip the header block, real columns start after

    df = df.rename(columns=RENAME)      # rename BY NAME, so Ludhiana's column order doesn't matter
    assert set(RENAME.values()) <= set(df.columns), f"Missing columns for {city}"

    df["time"] = pd.to_datetime(df["time"])
    df.insert(0, "country", CITY_COUNTRY[city])
    df.insert(0, "city", city)

    centroid = {"city": city, "lat": meta["latitude"].iloc[0], "lon": meta["longitude"].iloc[0]}
    return df, centroid

**Loader function.** This does not load anything yet. It defines a reusable recipe, `load_weather_file(city)`, that we run once per city in the next cell.

What the recipe does for one city, in order:
1. Finds that city's file and opens it (CSV or Excel).
2. Skips the first 3 rows of the file. They hold the city's coordinates and timezone, not weather data, so we read them separately and keep only the latitude and longitude.
3. Renames the long column names to short ones.
4. Adds two new columns, `city` and `country`, so we still know whose data each row is after we stack the cities together.

Example for Lahore:
- Column name before: `temperature_2m (°C)`. After: `temp_c`.
- Two new columns added to every row: `city = Lahore`, `country = PK`.
- Coordinates saved on the side: `lat = 31.529`, `lon = 74.390`.

In [31]:
frames, centroids = [], []
for city in WEATHER_FILES:
    df, c = load_weather_file(city)
    frames.append(df)
    centroids.append(c)
    print(f"{city:15s} {len(df):7d} rows")

weather_raw = pd.concat(frames, ignore_index=True)
city_centroids = pd.DataFrame(centroids)

print("\nTotal shape:", weather_raw.shape)
display(weather_raw.head())
display(city_centroids)

Lahore            40896 rows
Faisalabad        40896 rows
Multan            40896 rows
Rawalpindi        40896 rows
Sialkot           40896 rows
Rahim Yar Khan    40896 rows
Islamabad         40896 rows
Amritsar          40512 rows
Ferozepur         40512 rows
Ludhiana          25920 rows
Jalandhar         40512 rows

Total shape: (433728, 8)


,city,country,time,temp_c,wind_speed_kmh,wind_dir_deg,precip_mm,rh_pct
0,Lahore,PK,2022-01-12 00:00:00,6.0,7.6,301,0.0,99
1,Lahore,PK,2022-01-12 01:00:00,6.5,5.8,300,0.0,98
2,Lahore,PK,2022-01-12 02:00:00,6.1,7.4,309,0.0,97
3,Lahore,PK,2022-01-12 03:00:00,7.9,5.9,322,0.0,92
4,Lahore,PK,2022-01-12 04:00:00,10.2,4.2,329,0.0,81


,city,lat,lon
0,Lahore,31.528997,74.389950
1,Faisalabad,31.458698,73.118280
2,Multan,30.123022,71.495330
3,Rawalpindi,33.567660,73.048325
4,Sialkot,32.513180,74.562040
5,Rahim Yar Khan,28.435852,70.261360
6,Islamabad,33.637960,73.027300
7,Amritsar,31.669594,74.892080
8,Ferozepur,30.966606,74.644550
9,Ludhiana,30.896300,75.834300


**Load all 11 and stack.** This cell runs the recipe from Cell 3 once for each of the 11 cities. Each run gives back one small table for that city. Then we join all 11 tables, one under the other, into one big table called `weather_raw`.

It also builds `city_centroids`, a small table with each city's latitude and longitude. Person C will need these later to measure how far each fire is from each city.

Example of the stacking:
- Before: 11 separate tables, for example Lahore has 40,896 rows and Ludhiana has 25,920 rows.
- After: one table, `weather_raw`, with all of them combined (433,728 rows). The `city` column tells us which rows belong to which city.

### Step 1 summary
- Loaded all 11 weather files (10 CSV and 1 Excel) with one loader function.
- Gave every row a `city` and `country` so the cities can be stacked into one table.
- Result: `weather_raw` with 433,728 rows and 8 columns, plus `city_centroids` with each city's coordinates.

## Step 2: Check each file

In [32]:
value_cols = ["temp_c", "rh_pct", "precip_mm", "wind_speed_kmh", "wind_dir_deg"]

**Column list.** Saves the names of the 5 weather columns in one list, so we can reuse it in the checks below. It prints nothing.

In [33]:
rows = []
for city, d in weather_raw.groupby("city"):
    first = d["time"].min()
    last = d["time"].max()
    rows.append({
        "city": city,
        "rows": len(d),
        "first_time": first,
        "last_time": last,
        "expected_rows": int((last - first) / pd.Timedelta(hours=1)) + 1,
        "duplicate_times": d["time"].duplicated().sum(),
        "empty_values": int(d[value_cols].isna().sum().sum()),
    })

file_check = pd.DataFrame(rows)
file_check

,city,rows,first_time,last_time,expected_rows,duplicate_times,empty_values
0,Amritsar,40512,2022-01-30,2026-09-13 23:00:00,40512,0,0
1,Faisalabad,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
2,Ferozepur,40512,2022-01-30,2026-09-13 23:00:00,40512,0,0
3,Islamabad,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
4,Jalandhar,40512,2022-01-30,2026-09-13 23:00:00,40512,0,0
5,Lahore,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
6,Ludhiana,25920,2022-01-15,2024-12-29 23:00:00,25920,0,0
7,Multan,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
8,Rahim Yar Khan,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0
9,Rawalpindi,40896,2022-01-12,2026-09-11 23:00:00,40896,0,0


**Per-city check.** Goes through the stacked table one city at a time and builds a small summary table, one row per city. For each city it records:
- how many rows it has, and its first and last timestamp
- `expected_rows`: how many hourly rows there should be between the first and last time
- how many timestamps appear twice, and how many empty values there are

If `rows` equals `expected_rows`, no hours are skipped.

In [34]:
full_frames = []
for city, d in weather_raw.groupby("city"):
    d = d.set_index("time")
    all_hours = pd.date_range(d.index.min(), d.index.max(), freq="h")
    d = d.reindex(all_hours)          # missing hours appear as empty rows
    d["city"] = city
    d["country"] = CITY_COUNTRY[city]
    d.index.name = "time"
    full_frames.append(d.reset_index())

weather_full = pd.concat(full_frames, ignore_index=True)
weather_full = weather_full[weather_raw.columns]   # keep the same column order as before

print("Before:", weather_raw.shape)
print("After: ", weather_full.shape)

Before: (433728, 8)
After:  (433728, 8)


**Full hourly calendar.** For each city, this makes a complete hour-by-hour list from its first to its last time, then lines the city's data up against that list. If an hour was skipped in the file, it now appears as an empty row (NaN) instead of silently disappearing. Then it stacks the 11 cities back together as `weather_full`.

Example: if a city had rows for 00:00, 01:00 and 03:00, it now has 00:00, 01:00, 02:00 (empty) and 03:00.

If the shape before and after is the same, nothing was skipped.

In [35]:
weather_full[value_cols].describe().round(1)

,temp_c,rh_pct,precip_mm,wind_speed_kmh,wind_dir_deg
count,433728.0,433728.0,433728.0,433728.0,433728.0
mean,24.0,62.0,0.1,7.4,181.0
std,8.8,23.1,0.6,4.1,108.6
min,0.5,2.0,0.0,0.0,0.0
25%,17.3,45.0,0.0,4.5,87.0
50%,25.4,64.0,0.0,6.8,170.0
75%,30.5,81.0,0.0,9.4,289.0
max,49.9,100.0,27.8,42.7,360.0


**Value range check.** Shows the count, average, minimum and maximum of each weather column, so we can spot impossible values.

What to look for: `rh_pct` should stay between 0 and 100, and `wind_dir_deg` between 0 and 360. Rain and wind speed should never be negative.

### Step 2 summary
- Checked each city: no duplicate timestamps, no skipped hours, no empty values.
- Built a full hourly calendar per city. Nothing was added, so the data had no gaps.
- Checked value ranges: humidity 2 to 100, wind direction 0 to 360, and no negative rain or wind.
- Result: `weather_full` is a complete and clean hourly table (433,728 rows).

## Step 3: Convert to Pakistan time, split date and time, cut to 2022 to 2023

In [36]:
header = pd.read_csv(ROOT / weather_path("Lahore"), nrows=1)
header[["timezone", "utc_offset_seconds"]]

,timezone,utc_offset_seconds
0,GMT,0


**Confirm the timezone.** Before adding 5 hours, we check the file header to make sure the times really are GMT. If a file were already in Pakistan time, adding 5 hours would shift everything wrongly.


In [37]:
weather = weather_full.rename(columns={"time": "time_gmt"})
weather["datetime_pkt"] = weather["time_gmt"] + pd.Timedelta(hours=5)
weather["date"] = weather["datetime_pkt"].dt.normalize()   # calendar day, time set to 00:00
weather["hour"] = weather["datetime_pkt"].dt.hour          # 0 to 23

# tidy column order
weather = weather[["city", "country", "datetime_pkt", "date", "hour", "time_gmt"] + value_cols]
weather.head()

,city,country,datetime_pkt,date,hour,time_gmt,temp_c,rh_pct,precip_mm,wind_speed_kmh,wind_dir_deg
0,Amritsar,IN,2022-01-30 05:00:00,2022-01-30,5,2022-01-30 00:00:00,6.6,98,0.0,6.9,298
1,Amritsar,IN,2022-01-30 06:00:00,2022-01-30,6,2022-01-30 01:00:00,7.4,94,0.0,9.0,293
2,Amritsar,IN,2022-01-30 07:00:00,2022-01-30,7,2022-01-30 02:00:00,7.5,92,0.0,9.5,295
3,Amritsar,IN,2022-01-30 08:00:00,2022-01-30,8,2022-01-30 03:00:00,8.8,86,0.0,10.2,312
4,Amritsar,IN,2022-01-30 09:00:00,2022-01-30,9,2022-01-30 04:00:00,10.9,81,0.0,8.1,328


**Convert to a common clock and split.** Adds 5 hours to every timestamp for all 11 cities (GMT to UTC+5, which is Pakistan time). We use one common clock for the whole basin so weather, air quality and fires from all three team members line up on the same date. Indian cities are officially 30 minutes ahead of this, which we note as a limitation. Then it makes two new columns: `date` (the calendar day) and `hour` (0 to 23). The original GMT time is kept as `time_gmt` so we can always trace back.

Example: GMT `2022-03-01 20:00` becomes `2022-03-02 01:00`, so the date changes to March 2. This is why we convert before taking the date.

In [38]:
start = pd.Timestamp("2022-01-01")
end = pd.Timestamp("2023-12-31")

weather = weather[(weather["date"] >= start) & (weather["date"] <= end)].reset_index(drop=True)

print("Rows after cut:", len(weather))
weather.groupby("city").agg(
    rows=("datetime_pkt", "size"),
    first=("datetime_pkt", "min"),
    last=("datetime_pkt", "max"),
)

Rows after cut: 188393


,rows,first,last
city,,,
Amritsar,16819,2022-01-30 05:00:00,2023-12-31 23:00:00
Faisalabad,17251,2022-01-12 05:00:00,2023-12-31 23:00:00
Ferozepur,16819,2022-01-30 05:00:00,2023-12-31 23:00:00
Islamabad,17251,2022-01-12 05:00:00,2023-12-31 23:00:00
Jalandhar,16819,2022-01-30 05:00:00,2023-12-31 23:00:00
Lahore,17251,2022-01-12 05:00:00,2023-12-31 23:00:00
Ludhiana,17179,2022-01-15 05:00:00,2023-12-31 23:00:00
Multan,17251,2022-01-12 05:00:00,2023-12-31 23:00:00
Rahim Yar Khan,17251,2022-01-12 05:00:00,2023-12-31 23:00:00


**Cut to the 2022 to 2023 window.** Keeps only rows whose Pakistan date is from 1 Jan 2022 to 31 Dec 2023. Everything from 2024 onward is dropped here (2024 stays sealed for the final test). Then a per-city table shows the first and last time and row count after the cut.

Example: a row dated 2024-01-01 04:00 Pakistan time (which was 23:00 GMT on 31 Dec) is removed.

### Step 3 summary
- Confirmed the raw times are GMT, then added 5 hours to get Pakistan time.
- Created `date` (calendar day) and `hour` columns from the Pakistan time.
- Cut the table to 1 Jan 2022 to 31 Dec 2023. Dropped everything after, so 2024 stays untouched.
- Result: `weather`, the hourly table in Pakistan time for 2022 to 2023.

## Step 4: Missing data report

In [39]:
missing_overall = pd.DataFrame({
    "missing_count": weather[value_cols].isna().sum(),
    "missing_percent": (weather[value_cols].isna().mean() * 100).round(2),
})
missing_overall

,missing_count,missing_percent
temp_c,0,0.0
rh_pct,0,0.0
precip_mm,0,0.0
wind_speed_kmh,0,0.0
wind_dir_deg,0,0.0


**Missing values per column.** Counts how many values are empty in each weather column across the whole table, and what percent of the column that is.


In [40]:
missing_by_city = weather.groupby("city")[value_cols].apply(lambda d: d.isna().sum())
missing_by_city

,temp_c,rh_pct,precip_mm,wind_speed_kmh,wind_dir_deg
city,,,,,
Amritsar,0,0,0,0,0
Faisalabad,0,0,0,0,0
Ferozepur,0,0,0,0,0
Islamabad,0,0,0,0,0
Jalandhar,0,0,0,0,0
Lahore,0,0,0,0,0
Ludhiana,0,0,0,0,0
Multan,0,0,0,0,0
Rahim Yar Khan,0,0,0,0,0


**Missing values per city.** The same count, but split by city, so we can see if one city has a problem the overall table would hide. One row per city, one column per weather variable.

In [41]:
hours_per_day = weather.groupby(["city", "date"]).size().reset_index(name="hours")
partial_days = hours_per_day[hours_per_day["hours"] < 24]

print("Days with fewer than 24 hours:", len(partial_days))
partial_days

Days with fewer than 24 hours: 11


,city,date,hours
0,Amritsar,2022-01-30,19
701,Faisalabad,2022-01-12,19
1420,Ferozepur,2022-01-30,19
2121,Islamabad,2022-01-12,19
2840,Jalandhar,2022-01-30,19
3541,Lahore,2022-01-12,19
4260,Ludhiana,2022-01-15,19
4976,Multan,2022-01-12,19
5695,Rahim Yar Khan,2022-01-12,19
6414,Rawalpindi,2022-01-12,19


**Hours per day.** Counts how many hourly rows each city has on each date. A full day has 24. We list any day with fewer, because a daily average built from only a few hours would be misleading.

Example: a city whose first row is at 05:00 has only 19 hours (05:00 to 23:00) on its first day.

### Step 4 summary
- Counted missing values for all 5 weather columns, overall and per city. None are missing.
- Checked that every day has 24 hourly rows. Only the first day of each city is short (19 hours), because of the 5-hour shift to Pakistan time.
- Result: weather needs no filling or imputation. We will simply not use the 11 partial first days.

## Step 5: Wind processing

In [42]:
import numpy as np

angle_rad = np.deg2rad(weather["wind_dir_deg"])
weather["u_kmh"] = -weather["wind_speed_kmh"] * np.sin(angle_rad)
weather["v_kmh"] = -weather["wind_speed_kmh"] * np.cos(angle_rad)

weather[["wind_speed_kmh", "wind_dir_deg", "u_kmh", "v_kmh"]].head()

,wind_speed_kmh,wind_dir_deg,u_kmh,v_kmh
0,6.9,298,6.092338,-3.239354
1,9.0,293,8.284544,-3.516580
2,9.5,295,8.609924,-4.014873
3,10.2,312,7.580077,-6.825132
4,8.1,328,4.292346,-6.869190


**Split wind into U and V.** Turns each hourly wind (speed and direction) into two numbers: `u_kmh` (the east-west push, positive means blowing toward the east) and `v_kmh` (the north-south push, positive means blowing toward the north). The minus signs are there because Open-Meteo gives the direction the wind comes FROM, and we want where it blows TO.

Example: wind from the north (0°) at 10 km/h gives `u = 0` and `v = -10` (blowing south).

In [43]:
toy = pd.DataFrame({"from_deg": [0, 90, 180, 270], "speed": [10, 10, 10, 10]})
a = np.deg2rad(toy["from_deg"])
toy["u"] = -toy["speed"] * np.sin(a)
toy["v"] = -toy["speed"] * np.cos(a)
toy["back_to_deg"] = (np.degrees(np.arctan2(-toy["u"], -toy["v"])) + 360) % 360
toy.round(1)

,from_deg,speed,u,v,back_to_deg
0,0,10,-0.0,-10.0,0.0
1,90,10,-10.0,-0.0,90.0
2,180,10,-0.0,10.0,180.0
3,270,10,10.0,0.0,270.0


**Toy test.** Checks our conversion on 4 simple winds (from north, east, south, west). We turn each into U and V, then turn it back into an angle. If the formulas are right, the angle we get back is the same one we started with.

In [44]:
daily_wind = (
    weather.groupby(["city", "country", "date"])
    .agg(u_mean=("u_kmh", "mean"), v_mean=("v_kmh", "mean"), n_hours=("hour", "size"))
    .reset_index()
)

daily_wind["vec_len_kmh"] = np.hypot(daily_wind["u_mean"], daily_wind["v_mean"])
daily_wind["wind_dir_daily_deg"] = (
    np.degrees(np.arctan2(-daily_wind["u_mean"], -daily_wind["v_mean"])) + 360
) % 360

daily_wind.head()

,city,country,date,u_mean,v_mean,n_hours,vec_len_kmh,wind_dir_daily_deg
0,Amritsar,IN,2022-01-30,3.700656,-0.202292,19,3.706181,273.128897
1,Amritsar,IN,2022-01-31,-1.957755,3.416655,24,3.937809,150.187139
2,Amritsar,IN,2022-02-01,2.520272,-2.201640,24,3.346490,311.139568
3,Amritsar,IN,2022-02-02,-1.896699,-1.580685,24,2.469014,50.192592
4,Amritsar,IN,2022-02-03,-5.945905,-6.154257,24,8.557375,44.013525


**Daily wind per city.** Groups the hourly rows by city and date, averages U and V over the day, and then turns the daily averages back into two numbers:
- `vec_len_kmh`: how strong the average wind is (close to 0 means the wind kept changing direction or the air was nearly still)
- `wind_dir_daily_deg`: the direction the wind came FROM, as a compass angle (0 to 360)

`n_hours` counts how many hourly rows went into each day.

Example: on a day with half the hours from 350° and half from 10°, the plain average of the angles would say 180° (wrong). Averaging U and V correctly gives about 0° (north).

In [45]:
print("Rows:", len(daily_wind))
daily_wind[["vec_len_kmh", "wind_dir_daily_deg"]].describe().round(1)

Rows: 7852


,vec_len_kmh,wind_dir_daily_deg
count,7852.0,7852.0
mean,5.6,184.2
std,3.2,112.6
min,0.0,0.0
25%,3.2,87.0
50%,5.1,166.9
75%,7.4,302.6
max,22.7,360.0


**Quick check of the daily wind.** Prints how many city-days we have and summarizes the two new columns, to make sure nothing looks impossible.

What to look for: angles between 0 and 360, and wind strength never negative.

### Step 5 summary
- Split each hourly wind into U (horizontal component) and V (vertical component) parts.
- Tested the formulas on 4 simple winds: the angles came back correctly.
- Averaged U and V over each Pakistan day, then converted back to a daily wind direction (where the wind comes from) and a wind strength. -> This was the reason why we did this step.
- Result: `daily_wind`, one row per city per day, ready for the stagnation flag in Step 6.

## Step 6: Stagnation flag

In [46]:
import importlib, src.config
importlib.reload(src.config)
from src.config import TRAIN_START, TRAIN_END

train_days = daily_wind[daily_wind["date"].between(TRAIN_START, TRAIN_END)]
train_harvest = train_days[train_days["date"].dt.month.isin([10, 11, 12])]

print("Training city-days:", len(train_days))
print("Training Oct-Dec city-days:", len(train_harvest))

Training city-days: 4323
Training Oct-Dec city-days: 1012


**Pick out the training days.** Reloads the config (so it sees the new dates), then keeps only the daily rows inside the training period, and from those only the harvest months (Oct, Nov, Dec). We use only these days to judge our threshold, so nothing from the validation period influences it.


In [47]:
share = pd.DataFrame({
    f"below_{t}_kmh": train_harvest.groupby("city")["vec_len_kmh"].apply(lambda s: (s < t).mean() * 100)
    for t in [1, 2, 3]
}).round(1)
share

,below_1_kmh,below_2_kmh,below_3_kmh
city,,,
Amritsar,0.0,8.7,26.1
Faisalabad,1.1,6.5,13.0
Ferozepur,4.3,12.0,19.6
Islamabad,0.0,0.0,8.7
Jalandhar,5.4,18.5,31.5
Lahore,2.2,6.5,16.3
Ludhiana,6.5,12.0,27.2
Multan,3.3,8.7,26.1
Rahim Yar Khan,0.0,4.3,17.4


**Try three thresholds.** For each city, shows what percent of harvest-season training days would be flagged as stagnant if we used a cutoff of 1, 2 or 3 km/h. This shows how sensitive the flag is to the choice.


In [48]:
STAGNANT_KMH = 2
daily_wind["stagnant"] = (daily_wind["vec_len_kmh"] < STAGNANT_KMH).astype(int)

print("Days flagged overall:", daily_wind["stagnant"].sum(),
      f"({daily_wind['stagnant'].mean() * 100:.1f}%)")

train_harvest = daily_wind[daily_wind["date"].between(TRAIN_START, TRAIN_END)
                           & daily_wind["date"].dt.month.isin([10, 11, 12])]
train_harvest.groupby("city")["stagnant"].mean().mul(100).round(1)

Days flagged overall: 837 (10.7%)


city
Amritsar           8.7
Faisalabad         6.5
Ferozepur         12.0
Islamabad          0.0
Jalandhar         18.5
Lahore             6.5
Ludhiana          12.0
Multan             8.7
Rahim Yar Khan     4.3
Rawalpindi         5.4
Sialkot           13.0
Name: stagnant, dtype: float64

**Create the flag.** Sets the cutoff to 2 km/h and adds a `stagnant` column to the daily wind table: 1 if the daily wind strength is below 2, otherwise 0. The daily angle stays in the table, but on flagged days it should not be trusted. Then prints how many days are flagged overall, and the percent flagged per city in the training harvest season.

Example: a day with `vec_len_kmh = 1.3` gets `stagnant = 1`. A day with `vec_len_kmh = 4.2` gets `stagnant = 0`.

### Step 6 summary
- Looked at the training-period harvest season only (Oct to Dec 2022) to choose the cutoff.
- Compared three cutoffs (1, 2 and 3 km/h) to see how many days each would flag.
- Chose 2 km/h: below this the daily wind angle is too weak to trust. No city had more than 18.5% of its training harvest days flagged.
- Result: `daily_wind` now has a `stagnant` column (1 or 0). 837 of 7,852 city-days (10.7%) are flagged. Islamabad has none in the training harvest season.

## Step 7: Save the daily wind table

In [49]:
CLEAN_DIR = ROOT / "Dataset" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

daily_wind.to_parquet(CLEAN_DIR / "wind_daily.parquet", index=False)
print("Saved to:", CLEAN_DIR / "wind_daily.parquet")

Saved to: d:\Habib Courses material\Data Science\Project\Smog-Analysis-DSSG\Dataset\clean\wind_daily.parquet


**Save to file.** Creates a `Dataset/clean/` folder (it's inside `Dataset/`, so Git ignores it) and saves `daily_wind` there as `wind_daily.parquet`. Parquet keeps the date and number types exactly as they are, which CSV would not.

Columns Person C needs: `city`, `country`, `date`, `wind_dir_daily_deg`, `vec_len_kmh`, `stagnant`.

In [50]:
check = pd.read_parquet(CLEAN_DIR / "wind_daily.parquet")

print("Shape:", check.shape)
print("Cities:", check["city"].nunique())
print("Duplicate city-date rows:", check.duplicated(["city", "date"]).sum())
print("Date range:", check["date"].min().date(), "to", check["date"].max().date())
print("Stagnant share:", round(check["stagnant"].mean() * 100, 1), "%")
check.dtypes

Shape: (7852, 9)
Cities: 11
Duplicate city-date rows: 0
Date range: 2022-01-12 to 2023-12-31
Stagnant share: 10.7 %


city                             str
country                          str
date                  datetime64[us]
u_mean                       float64
v_mean                       float64
n_hours                        int64
vec_len_kmh                  float64
wind_dir_daily_deg           float64
stagnant                       int64
dtype: object

**Read it back and check.** Opens the saved file again and checks the things the team agreed on: right number of rows, no city-date appearing twice, the date range, and the share of flagged days. If this passes, the file is safe to share.

Example: `Duplicate city-date rows: 0`

### Step 7 summary
- Saved the daily wind table as `wind_daily.parquet` in `Dataset/clean/`.
- Read it back and checked: 11 cities, no duplicate city-dates, correct date range.
- Result: the file is ready to hand to Person C for the wind cone.